In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")

df = pd.read_csv("../data/raw/stock_market_daily.csv", parse_dates=["date"])
sectors = ["Financials", "Technology", "ConsumerStaples", "Utilities", "Energy"]
data = df[["date", "year", "crisis_gfc"] + sectors].dropna().set_index("date")

before = data.loc["2003-01-01":"2007-10-08"]
during = data.loc["2007-10-09":"2009-03-09"]
after  = data.loc["2009-03-10":"2010-12-31"]

data.head()

,year,crisis_gfc,Financials,Technology,ConsumerStaples,Utilities,Energy
date,,,,,,,
2000-01-04,2000,0,-0.047966,-0.056507,-0.024891,0.010591,-0.009756
2000-01-05,2000,0,-0.009918,-0.005338,-0.011764,0.037025,0.002781
2000-01-06,2000,0,0.041497,-0.040156,0.019411,0.002855,0.047487
2000-01-07,2000,0,0.008738,0.034152,0.049830,0.010770,0.012612
2000-01-10,2000,0,-0.004872,0.047272,-0.006921,0.002564,-0.012649


In [3]:
weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])  

portfolio_returns_full = data[sectors].dot(weights)
portfolio_returns_before = before[sectors].dot(weights)
portfolio_returns_during = during[sectors].dot(weights)
portfolio_returns_after = after[sectors].dot(weights)

portfolio_returns_during.describe()

count    356.000000
mean      -0.001426
std        0.025120
min       -0.096941
25%       -0.013619
50%        0.000620
75%        0.008402
max        0.133362
dtype: float64

In [5]:
def historical_var(returns, cl=0.95):
    return -np.percentile(returns, 100 * (1 - cl))

def historical_cvar(returns, cl=0.95):
    var = historical_var(returns, cl)
    tail_losses = returns[returns <= -var]
    return -tail_losses.mean()

for period_name, r in [("Before", portfolio_returns_before),
                        ("During", portfolio_returns_during),
                        ("After", portfolio_returns_after)]:
    for cl in [0.95, 0.99]:
        var = historical_var(r, cl)
        cvar = historical_cvar(r, cl)
        print(period_name + " | CL=" + str(cl*100) + "% | VaR=" + str(round(var*100, 2)) + "%")
    print()

Before | CL=95.0% | VaR=1.3%
Before | CL=99.0% | VaR=1.87%

During | CL=95.0% | VaR=4.19%
During | CL=99.0% | VaR=7.39%

After | CL=95.0% | VaR=2.13%
After | CL=99.0% | VaR=3.19%



In [10]:
for period_name, r in [("Before", portfolio_returns_before),
                        ("During", portfolio_returns_during),
                        ("After", portfolio_returns_after)]:
    mu, sigma = r.mean(), r.std()
    for cl in [0.95, 0.99]:
        z = stats.norm.ppf(cl)
        param_var = -(mu - z * sigma)
        hist_var = historical_var(r, cl)
        gap = (param_var / hist_var - 1) if hist_var != 0 else float("nan")
        print(period_name + " | CL=" + str(cl) + " | Parametric VaR=" + str(round(param_var, 4)) + " | Historical VaR=" + str(round(hist_var, 4)) + " | Gap=" + str(round(gap, 4)))
    print()

Before | CL=0.95 | Parametric VaR=0.0123 | Historical VaR=0.013 | Gap=-0.0515
Before | CL=0.99 | Parametric VaR=0.0178 | Historical VaR=0.0187 | Gap=-0.0488

During | CL=0.95 | Parametric VaR=0.0427 | Historical VaR=0.0419 | Gap=0.0211
During | CL=0.99 | Parametric VaR=0.0599 | Historical VaR=0.0739 | Gap=-0.1903

After | CL=0.95 | Parametric VaR=0.0209 | Historical VaR=0.0213 | Gap=-0.0201
After | CL=0.99 | Parametric VaR=0.0302 | Historical VaR=0.0319 | Gap=-0.0527



In [13]:
from pypfopt import EfficientCVaR

returns_matrix_during = during[sectors]
mu_during = returns_matrix_during.mean() * 252  

ec = EfficientCVaR(mu_during, returns_matrix_during)
ec.min_cvar()
cvar_weights = ec.clean_weights()
print("Min-CVaR weights (during crisis):", cvar_weights)

cvar_value = ec.portfolio_performance(verbose=True)

Min-CVaR weights (during crisis): OrderedDict({'Financials': 0.0, 'Technology': 0.0, 'ConsumerStaples': 0.90274, 'Utilities': 0.09726, 'Energy': 0.0})
Expected annual return: -18.5%
Conditional Value at Risk: 3.55%


equal_weight_cvar_during = historical_cvar(portfolio_returns_during, cl=0.95)
print(f"Equal-weight 95% CVaR (during crisis): {equal_weight_cvar_during }")
print(f"Min-CVaR optimized 95% CVaR (during crisis): 3.55%")

# How can I hedge if I hold a position on Financials during the crisis?

In [20]:
from scipy import stats as st

financials_returns = during["Financials"]
financials_price = 100 * (1 + financials_returns).cumprod()

S0 = financials_price.iloc[-1]          
K = S0 * 0.95                   
T = 30 / 252                         
r = 0.02                          
sigma = financials_returns.std() * np.sqrt(252)

def black_scholes_put(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    put_price = K*np.exp(-r*T)*st.norm.cdf(-d2) - S*st.norm.cdf(-d1)
    delta = st.norm.cdf(d1) - 1
    return put_price, delta

put_price, put_delta = black_scholes_put(S0, K, T, r, sigma)
print(f"Financials virtual spot: {S0}, Strike: {K}, Vol: {sigma}")
print(f"Put price: {put_price}, Put delta: {put_delta}")
print(f"Hedge ratio (puts per share to delta-hedge): {-1/put_delta}")

Financials virtual spot: 24.973579417753687, Strike: 23.724900446866002, Vol: 0.768278657627298
Put price: 1.964010920457481, Put delta: -0.368803924009772
Hedge ratio (puts per share to delta-hedge): 2.7114678963489105
